low api pytorch operation 

In [11]:
import random as rand
from typing import cast

import torch
from sklearn.model_selection import train_test_split

INPUT_NUMBER = 10
DATASET_SIZE = 100
torch.manual_seed(1)
rand.seed(1)
# create dataset
#
X = []
Y = []
for _ in range(DATASET_SIZE):
    x = float(rand.randint(1, 100))
    X.append(x)
    Y.append(x * 5 + rand.randint(0, 3))

x_all = torch.tensor(X, dtype=torch.float32)
y_all = torch.tensor(Y, dtype=torch.float32)

x_train_raw, x_temp_raw, y_train, y_temp =cast(
    tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor],
    train_test_split(x_all, y_all, test_size=0.30, random_state=1)
)
x_val_raw, x_test_raw, y_val, y_test =cast(
    tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor],
    train_test_split(x_temp_raw, y_temp, test_size=0.50, random_state=1)
)


x_mean, x_std = x_train_raw.mean(), x_train_raw.std()
x_train = (x_train_raw - x_mean) / x_std
x_val   = (x_val_raw   - x_mean) / x_std
x_test  = (x_test_raw  - x_mean) / x_std



lr = 0.1
n_epoch = 50
train_losses = []
val_losses = []

w = torch.randn(1 , 1, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

x_train = x_train.reshape(-1 , 1 )
y_train = y_train.reshape(-1 , 1 )
print(w.shape)
# w = w.reshape(1,1)
print(w.shape)


# no early stopping yer , but i will consider it in mind for future
for epoch in range(n_epoch):
    z = x_train @ w + b
    loss = ((z - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        gw, gb = w.grad, b.grad
        assert gw is not None and gb is not None
        w -= lr * gw
        b -= lr * gb
        gw.zero_()
        gb.zero_()
    # Validate 
    with torch.no_grad():
        loss_val = (((x_val * w + b)-y_val)**2).mean()
        val_losses.append(loss_val)

    train_losses.append(loss)
    print(f"Epoch {epoch+1} | loss {loss.item()} | validation loss {val_losses[epoch].item()}")

test_val = (((x_test * w + b)-y_test)**2).mean()
print(f"Test Validation  {test_val}")

checkpoint = {
    "state_dict": {"w": w.detach(), "b": b.detach()},
    "scaler": {"mean": x_mean, "std": x_std},
    "config": {"lr": lr, "epoch": n_epoch, "model": "linear_1d", "n_train": len(x_train), "n_test": len(x_test)},
    "metrics": {
        "train": [t.item() for t in train_losses],
        "val":   [t.item() for t in val_losses],
        "test":  test_val.item(),
    },
    "version": "1.0.0"
}
torch.save(checkpoint, "model_v1.0.0.pt")
print(f"checkpoint \n {checkpoint}")

torch.Size([1, 1])
torch.Size([1, 1])
Epoch 1 | loss 85085.3984375 | validation loss 66656.546875
Epoch 2 | loss 54543.734375 | validation loss 42756.5859375
Epoch 3 | loss 34965.5625 | validation loss 27427.30078125
Epoch 4 | loss 22415.2265625 | validation loss 17594.93359375
Epoch 5 | loss 14369.916015625 | validation loss 11288.150390625
Epoch 6 | loss 9212.478515625 | validation loss 7242.64013671875
Epoch 7 | loss 5906.27978515625 | validation loss 4647.5185546875
Epoch 8 | loss 3786.80810546875 | validation loss 2982.7060546875
Epoch 9 | loss 2428.086181640625 | validation loss 1914.6351318359375
Epoch 10 | loss 1557.04638671875 | validation loss 1229.35693359375
Epoch 11 | loss 998.6429443359375 | validation loss 789.637451171875
Epoch 12 | loss 640.658935546875 | validation loss 507.4532775878906
Epoch 13 | loss 411.1589050292969 | validation loss 326.34033203125
Epoch 14 | loss 264.0273742675781 | validation loss 210.07798767089844
Epoch 15 | loss 169.70138549804688 | validat

# High Api torch 

In [12]:
import random as rand
from types import CellType
from typing import cast

import torch 
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.nn.modules import ReLU

INPUT_NUMBER = 10
DATASET_SIZE = 100

lr = 0.1
n_epoch = 50
train_losses = []
val_losses = []

torch.manual_seed(1)
rand.seed(1)


# create dataset
#
X = []
Y = []
for _ in range(DATASET_SIZE):
    x = float(rand.randint(1, 100))
    X.append(x)
    Y.append(x * 5 + rand.randint(0, 3))

x_all = torch.tensor(X, dtype=torch.float32)
y_all = torch.tensor(Y, dtype=torch.float32)

x_train_raw, x_temp_raw, y_train, y_temp =cast(
    tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor],
    train_test_split(x_all, y_all, test_size=0.30, random_state=1)
)
x_val_raw, x_test_raw, y_val, y_test =cast(
    tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor],
    train_test_split(x_temp_raw, y_temp, test_size=0.50, random_state=1)
)


x_mean, x_std = x_train_raw.mean(), x_train_raw.std()
x_train = (x_train_raw - x_mean) / x_std
x_val   = (x_val_raw   - x_mean) / x_std
x_test  = (x_test_raw  - x_mean) / x_std


model = nn.Linear(in_features=1 , out_features=1)

In [13]:
# read the book somewhere around page 333 
x_train = x_train.reshape(-1,1)
y_train = y_train.reshape(-1,1)
x_val = x_val.reshape(-1,1) 
y_val = y_val.reshape(-1,1) 
x_test = x_test.reshape(-1,1) 
y_test = y_test.reshape(-1,1) 
print("train:", x_train.shape, "val:", x_val.shape, "y_train:", y_train.shape)
model(x_train[:2])

train: torch.Size([70, 1]) val: torch.Size([15, 1]) y_train: torch.Size([70, 1])


tensor([[-0.3131],
        [-1.1927]], grad_fn=<AddmmBackward0>)

In [14]:
optimizer = torch.optim.SGD(model.parameters() , lr=lr)
mse = nn.MSELoss()

train_losses, val_losses = [], []
for epoch in range(n_epoch):
    y_pred = model(x_train)
    loss = mse( y_pred ,  y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    with torch.no_grad():
        loss_val = mse(model(x_val) , y_val)
        val_losses.append(loss_val)

    train_losses.append(loss)
    print(f"Epoch {epoch+1} | loss {loss.item()} | validation loss {val_losses[epoch].item()}")

test_val =  mse(model(x_test)  , y_test)
print(f"Test Validation  {test_val}")



Epoch 1 | loss 85352.2890625 | validation loss 66859.3515625
Epoch 2 | loss 54714.73046875 | validation loss 42886.58984375
Epoch 3 | loss 35075.12109375 | validation loss 27510.646484375
Epoch 4 | loss 22485.419921875 | validation loss 17648.3671875
Epoch 5 | loss 14414.8876953125 | validation loss 11322.4091796875
Epoch 6 | loss 9241.2939453125 | validation loss 7264.6044921875
Epoch 7 | loss 5924.74169921875 | validation loss 4661.6015625
Epoch 8 | loss 3798.636962890625 | validation loss 2991.736083984375
Epoch 9 | loss 2435.6640625 | validation loss 1920.4261474609375
Epoch 10 | loss 1561.90234375 | validation loss 1233.0699462890625
Epoch 11 | loss 1001.7534790039062 | validation loss 792.019287109375
Epoch 12 | loss 642.6520385742188 | validation loss 508.9819641113281
Epoch 13 | loss 412.4364318847656 | validation loss 327.32183837890625
Epoch 14 | loss 264.8464050292969 | validation loss 210.70822143554688
Epoch 15 | loss 170.22633361816406 | validation loss 135.8343048095703


In [15]:
model = nn.Sequential(
    nn.Linear(1,10),
    nn.ReLU(),
    nn.Linear(10,5),
    nn.ReLU(),
    nn.Linear(5,1)
)

optimizer = torch.optim.SGD(model.parameters() , lr=0.02)
mse = nn.MSELoss()

train_losses, val_losses = [], []
for epoch in range(n_epoch):
    y_pred = model(x_train)
    loss = mse( y_pred ,  y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    with torch.no_grad():
        loss_val = mse(model(x_val) , y_val)
        val_losses.append(loss_val)

    train_losses.append(loss)
    print(f"Epoch {epoch+1} | loss {loss.item()} | validation loss {val_losses[epoch].item()}")

test_val =  mse(model(x_test)  , y_test)
print(f"Test Validation  {test_val}")




Epoch 1 | loss 85316.03125 | validation loss 94910.0546875
Epoch 2 | loss 77147.890625 | validation loss 472547488.0
Epoch 3 | loss 374796000.0 | validation loss 856869.3125
Epoch 4 | loss 795066.75 | validation loss 1049788791390208.0
Epoch 5 | loss 1924243193331712.0 | validation loss 1492521451520.0
Epoch 6 | loss 1492449361920.0 | validation loss 1375510593536.0
Epoch 7 | loss 1375441387520.0 | validation loss 1267673464832.0
Epoch 8 | loss 1267607011328.0 | validation loss 1168290217984.0
Epoch 9 | loss 1168226385920.0 | validation loss 1076698742784.0
Epoch 10 | loss 1076637466624.0 | validation loss 992287850496.0
Epoch 11 | loss 992229261312.0 | validation loss 914494849024.0
Epoch 12 | loss 914438488064.0 | validation loss 842800562176.0
Epoch 13 | loss 842746494976.0 | validation loss 776727101440.0
Epoch 14 | loss 776675196928.0 | validation loss 715833671680.0
Epoch 15 | loss 715783864320.0 | validation loss 659714146304.0
Epoch 16 | loss 659666239488.0 | validation loss 60

In [16]:
#
model = nn.Sequential(
    nn.Linear(1, 10),
    nn.ReLU(),
    nn.Linear(10, 1)
)
y_mean = y_train.mean()
y_std = y_train.std()

y_train_norm = (y_train - y_mean) / y_std
y_val_norm = (y_val - y_mean) / y_std
y_test_norm = (y_test - y_mean) / y_std

optimizer = torch.optim.SGD(model.parameters() , lr=0.02)
mse = nn.MSELoss()

val_losses, train_losses = [], []

for epoch in range(n_epoch):

    optimizer.zero_grad()

    y_pred = model(x_train)
    loss = mse(y_pred, y_train_norm)

    loss.backward()

    optimizer.step()

    with torch.no_grad():
        loss_val = mse(model(x_val), y_val_norm)

    print(
        f"Epoch {epoch + 1} | "
        f"Train: {loss.item():.6f} | "
        f"Validation: {loss_val.item():.6f}"
    )
test_val =  mse(model(x_test)  , y_test)

Epoch 1 | Train: 1.420167 | Validation: 1.356007
Epoch 2 | Train: 1.286562 | Validation: 1.246020
Epoch 3 | Train: 1.170237 | Validation: 1.147074
Epoch 4 | Train: 1.067484 | Validation: 1.057086
Epoch 5 | Train: 0.975616 | Validation: 0.974577
Epoch 6 | Train: 0.892743 | Validation: 0.898542
Epoch 7 | Train: 0.817467 | Validation: 0.828189
Epoch 8 | Train: 0.748692 | Validation: 0.762868
Epoch 9 | Train: 0.685557 | Validation: 0.701945
Epoch 10 | Train: 0.627578 | Validation: 0.646675
Epoch 11 | Train: 0.574834 | Validation: 0.595800
Epoch 12 | Train: 0.527116 | Validation: 0.549249
Epoch 13 | Train: 0.482891 | Validation: 0.505880
Epoch 14 | Train: 0.441963 | Validation: 0.465585
Epoch 15 | Train: 0.404335 | Validation: 0.428181
Epoch 16 | Train: 0.369709 | Validation: 0.393491
Epoch 17 | Train: 0.337854 | Validation: 0.361277
Epoch 18 | Train: 0.308447 | Validation: 0.331466
Epoch 19 | Train: 0.281377 | Validation: 0.303891
Epoch 20 | Train: 0.256460 | Validation: 0.278386
Epoch 21 

In [17]:

model = nn.Sequential(
    nn.Linear(1,10),
    nn.ReLU(),
    nn.Linear(10,5),
    nn.ReLU(),
    nn.Linear(5,1)
)

optimizer = torch.optim.SGD(model.parameters() , lr=0.02)
mse = nn.MSELoss()
loss_fn = lambda pred, target: torch.sqrt(nn.functional.mse_loss(pred, target))
#loss_fn = nn.HuberLoss(delta=140.0)

train_losses, val_losses = [], []
for epoch in range(n_epoch):
    y_pred = model(x_train)
    loss = loss_fn( y_pred ,  y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    with torch.no_grad():
        loss_val = loss_fn(model(x_val) , y_val)
        val_losses.append(loss_val)

    train_losses.append(loss)
    print(f"Epoch {epoch+1} | loss {loss.item()} | validation loss {val_losses[epoch].item()}")

test_val =  loss_fn(model(x_test)  , y_test)
print(f"Test Validation  {test_val}")

Epoch 1 | loss 291.64208984375 | validation loss 322.4266662597656
Epoch 2 | loss 291.61962890625 | validation loss 322.4036560058594
Epoch 3 | loss 291.5967712402344 | validation loss 322.380126953125
Epoch 4 | loss 291.5733947753906 | validation loss 322.35601806640625
Epoch 5 | loss 291.5494689941406 | validation loss 322.3313293457031
Epoch 6 | loss 291.5249938964844 | validation loss 322.3059997558594
Epoch 7 | loss 291.4999084472656 | validation loss 322.2799072265625
Epoch 8 | loss 291.4740905761719 | validation loss 322.2530517578125
Epoch 9 | loss 291.44757080078125 | validation loss 322.22540283203125
Epoch 10 | loss 291.4202575683594 | validation loss 322.1968078613281
Epoch 11 | loss 291.39208984375 | validation loss 322.1672668457031
Epoch 12 | loss 291.36297607421875 | validation loss 322.13665771484375
Epoch 13 | loss 291.3328552246094 | validation loss 322.10491943359375
Epoch 14 | loss 291.3016662597656 | validation loss 322.07196044921875
Epoch 15 | loss 291.269317626

nonsequesntial models 

In [18]:

# this model is just like regular 1/2 layer deep models + a concat layer 
# it work like this : 
# output(concat (ehidden2(hiden1(input)) , input ))
class WideAndDeep(nn.Module):
    def __init__(self, n_features ) -> None:
        super().__init__()
        self.deep_part = nn.Sequential(
            nn.Linear(1,10),
            nn.ReLU(),
            nn.Linear(10,5),
            nn.ReLU(),
            nn.Linear(5,5)
        )
        self.output_layer = nn.Linear(5 +n_features , 1)
    def forward(self , X):
        deep_output = self.deep_part(X)
        concatination = torch.concat([X , deep_output] , dim=1)
        return self.output_layer(concatination)


model = WideAndDeep(1)
optimizer = torch.optim.SGD(model.parameters() , lr=0.05)
loss_fn = lambda pred, target: torch.sqrt(nn.functional.mse_loss(pred, target))
#loss_fn = nn.MSELoss()
#loss_fn = nn.HuberLoss(delta=140)

val_losses, train_losses = [], []
for epoch in range(n_epoch):
    y_pred = model(x_train)
    loss = loss_fn( y_pred ,  y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    with torch.no_grad():
        loss_val = loss_fn(model(x_val) , y_val)
        val_losses.append(loss_val)

    train_losses.append(loss)
    print(f"Epoch {epoch+1} | loss {loss.item()} | validation loss {val_losses[epoch].item()}")

test_val =  loss_fn(model(x_test)  , y_test)
print(f"Test Validation  {test_val}")

Epoch 1 | loss 292.3302307128906 | validation loss 323.0733947753906
Epoch 2 | loss 292.2626647949219 | validation loss 322.9990234375
Epoch 3 | loss 292.1946716308594 | validation loss 322.923828125
Epoch 4 | loss 292.1259765625 | validation loss 322.8475341796875
Epoch 5 | loss 292.0562744140625 | validation loss 322.7697448730469
Epoch 6 | loss 291.9851989746094 | validation loss 322.6900939941406
Epoch 7 | loss 291.91229248046875 | validation loss 322.60809326171875
Epoch 8 | loss 291.837158203125 | validation loss 322.52313232421875
Epoch 9 | loss 291.7593078613281 | validation loss 322.4346618652344
Epoch 10 | loss 291.6781311035156 | validation loss 322.3420104980469
Epoch 11 | loss 291.5930480957031 | validation loss 322.2442932128906
Epoch 12 | loss 291.50311279296875 | validation loss 322.140380859375
Epoch 13 | loss 291.407470703125 | validation loss 322.0290222167969
Epoch 14 | loss 291.3049011230469 | validation loss 321.9085998535156
Epoch 15 | loss 291.1939697265625 | va

In [19]:
model = WideAndDeep(1)
optimizer = torch.optim.SGD(model.parameters() , lr=0.05)
loss_fn = lambda pred, target: torch.sqrt(nn.functional.mse_loss(pred, target))
#loss_fn = nn.MSELoss()
#loss_fn = nn.HuberLoss(delta=140)

val_losses, train_losses = [], []

y_mean = y_train.mean()
y_std = y_train.std()

y_train_norm = (y_train - y_mean) / y_std
y_val_norm = (y_val - y_mean) / y_std
y_test_norm = (y_test - y_mean) / y_std

for epoch in range(n_epoch):

    optimizer.zero_grad()

    y_pred = model(x_train)
    loss = loss_fn(y_pred, y_train_norm)

    loss.backward()

    optimizer.step()

    with torch.no_grad():
        loss_val = loss_fn(model(x_val), y_val_norm)

    print(
        f"Epoch {epoch + 1} | "
        f"Train: {loss.item():.6f} | "
        f"Validation: {loss_val.item():.6f}"
    )
test_val =  loss_fn(model(x_test)  , y_test)



Epoch 1 | Train: 0.832165 | Validation: 0.868156
Epoch 2 | Train: 0.779777 | Validation: 0.809327
Epoch 3 | Train: 0.727477 | Validation: 0.750624
Epoch 4 | Train: 0.675215 | Validation: 0.692045
Epoch 5 | Train: 0.622977 | Validation: 0.633548
Epoch 6 | Train: 0.570737 | Validation: 0.575452
Epoch 7 | Train: 0.518511 | Validation: 0.517839
Epoch 8 | Train: 0.466491 | Validation: 0.460392
Epoch 9 | Train: 0.414612 | Validation: 0.402996
Epoch 10 | Train: 0.362723 | Validation: 0.345708
Epoch 11 | Train: 0.310886 | Validation: 0.288477
Epoch 12 | Train: 0.259010 | Validation: 0.231312
Epoch 13 | Train: 0.207085 | Validation: 0.174250
Epoch 14 | Train: 0.155140 | Validation: 0.117447
Epoch 15 | Train: 0.103314 | Validation: 0.061618
Epoch 16 | Train: 0.052302 | Validation: 0.016813
Epoch 17 | Train: 0.014552 | Validation: 0.016718
Epoch 18 | Train: 0.013709 | Validation: 0.014980
Epoch 19 | Train: 0.014919 | Validation: 0.043089
Epoch 20 | Train: 0.037276 | Validation: 0.046432
Epoch 21 